<a href="https://colab.research.google.com/github/liuhuhuhuhu/financial-document-intelligence/blob/main/02_text_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Document Intelligence System

## 02. Text Chunking & Document Segmentation

**Project:** Financial Document Intelligence System  
**Document:** Apple Inc. FY2025 Form 10-K  
**Tools:** Python, PyMuPDF, Pandas, JSONL

### Overview

This notebook implements the second stage of an end-to-end financial document intelligence pipeline: **text chunking**.

Building on the PDF parsing module, the extracted text is divided into smaller, overlapping segments while preserving document metadata and page-level references.

This preprocessing step prepares financial reports for semantic retrieval and Retrieval-Augmented Generation (RAG).

### Objectives

1. Implement a character-based text chunking algorithm with configurable chunk size and overlap.
2. Preserve document identifiers, page numbers, and chunk-level metadata.
3. Evaluate chunk length distributions and perform basic quality checks.
4. Compare different chunking configurations to understand the trade-offs between retrieval granularity and contextual completeness.
5. Export structured text chunks in JSONL format for downstream embedding generation.

### Methodology

The initial implementation uses a **fixed-size, overlapping character-based chunking strategy**.

| Parameter | Value |
|---|---|
| Chunk Size | 1,000 characters |
| Chunk Overlap | 200 characters |
| Processing Unit | Individual PDF pages |
| Output Format | JSONL |
| Metadata | Document name, page number, chunk ID |

Text is processed page by page to maintain traceability to the original financial document. Overlapping windows help preserve context across adjacent chunks.

### Expected Output

A structured collection of text chunks, each containing its source document, page number, unique identifier, and extracted content.

These chunks will serve as inputs for the next stage: **text embeddings and vector-based semantic search**.

### Limitations

Fixed-size character splitting may divide sentences, financial tables, or numerical disclosures across chunks. Future iterations will explore sentence-aware, token-aware, and table-aware chunking strategies to improve retrieval quality.

In [ ]:

print("Number of pages:", len(pages))
print("First page:", pages[0]["page"])
print("Preview:", pages[0]["text"][:300])


Number of pages: 80
First page: 1
Preview: UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 27, 2025
or
☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURIT


In [ ]:

def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into overlapping character chunks."""

    if not 0 <= overlap < chunk_size:
        raise ValueError(
            "Overlap must be >= 0 and smaller than chunk_size."
        )

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


In [ ]:

all_chunks = []

for page in pages:
    page_chunks = chunk_text(
        page["text"],
        chunk_size=1000,
        overlap=200
    )

    for i, text in enumerate(page_chunks):
        all_chunks.append({
            "chunk_id": f"apple_2025_p{page['page']}_c{i+1}",
            "document": "Apple 2025 Form 10-K",
            "page": page["page"],
            "text": text
        })

print("Total chunks:", len(all_chunks))
print("First chunk:\n")
print(all_chunks[0])


Total chunks: 368
First chunk:

{'chunk_id': 'apple_2025_p1_c1', 'document': 'Apple 2025 Form 10-K', 'page': 1, 'text': 'UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\n☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended September\xa027, 2025\nor\n☐    TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the transition period from \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 to \xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0.\nCommission File Number: 001-36743\nApple Inc.\n(Exact name of Registrant as specified in its charter)\nCalifornia\n94-2404110\n(State or other jurisdiction\nof incorporation or organization)\n(I.R.S. Employer Identification No.)\nOne Apple Park Way\nCupertino, California\n95014\n(Address of principal executive offices)\n(Zip Code)\n(408) 996-1010\n(Registrant’s telephone number, including area code)\nSecu

In [ ]:

import pandas as pd

df_chunks = pd.DataFrame(all_chunks)

df_chunks["char_count"] = (
    df_chunks["text"].str.len()
)

display(
    df_chunks[
        ["chunk_id", "page", "char_count"]
    ].head(10)
)

print("\nChunk statistics:")
print(df_chunks["char_count"].describe())

assert df_chunks["char_count"].max() <= 1000
assert df_chunks["text"].str.len().min() > 0

print("\nQuality checks passed!")


,chunk_id,page,char_count
0,apple_2025_p1_c1,1,1000
1,apple_2025_p1_c2,1,900
2,apple_2025_p2_c1,2,1000
3,apple_2025_p2_c2,2,1000
4,apple_2025_p2_c3,2,1000
5,apple_2025_p2_c4,2,1000
6,apple_2025_p2_c5,2,350
7,apple_2025_p3_c1,3,999
8,apple_2025_p3_c2,3,551
9,apple_2025_p4_c1,4,1000



Chunk statistics:
count     368.000000
mean      910.255435
std       202.795208
min       207.000000
25%       998.000000
50%      1000.000000
75%      1000.000000
max      1000.000000
Name: char_count, dtype: float64

Quality checks passed!


In [ ]:

import json

output_path = "apple_2025_chunks.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(
            json.dumps(chunk, ensure_ascii=False) + "\n"
        )

print("Saved:", output_path)


Saved: apple_2025_chunks.jsonl


In [ ]:

results = []

for size in [500, 1000, 1500]:
    overlap = int(size * 0.2)

    count = sum(
        len(chunk_text(p["text"], size, overlap))
        for p in pages
    )

    results.append({
        "chunk_size": size,
        "overlap": overlap,
        "total_chunks": count
    })

display(pd.DataFrame(results))


,chunk_size,overlap,total_chunks
0,500,100,714
1,1000,200,368
2,1500,300,247
